# Fine-tuning NLLB-200 pour la traduction Français ↔ Fulfulde (Adamawa, Cameroun)

**Projet de démonstration de faisabilité** — corpus biblique (4680 paires de phrases alignées).

Ce notebook fait tout le pipeline :
1. Connexion à Google Drive (sauvegarde + reprise automatique après interruption)
2. Validation des données (non bloquante)
3. Split train/val/test
4. Fine-tuning bidirectionnel (FR→FF et FF→FR) de `facebook/nllb-200-distilled-600M`
5. Évaluation (BLEU / chrF++)
6. Sauvegarde finale + démo d'inférence

⚠️ **Limites à garder en tête** (voir README du dépôt pour plus de détails) :
- NLLB-200 ne contient pas de code dédié pour le fulfulde Adamawa Cameroun spécifiquement —
  on réutilise le code `fuv_Latn` (Nigerian Fulfulde), la variété la plus proche disponible.
  Le fine-tuning va adapter ce code vers ton dialecte, mais une validation par locuteur natif
  reste indispensable.
- Le corpus est 100% biblique : le modèle sera bon sur ce registre, moins fiable hors domaine.
- Ce notebook est conçu pour un GPU Colab gratuit (T4). Les temps indiqués sont approximatifs.


## 1. Connexion à Google Drive et configuration des chemins

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Dossier racine du projet sur ton Drive ===
# Tout (checkpoints, données traitées, modèle final, rapports) est sauvegardé ici,
# donc une interruption Colab ne fait rien perdre.
PROJECT_DIR = "/content/drive/MyDrive/fr-fulfulde-mt"

DATA_RAW_DIR = os.path.join(PROJECT_DIR, "data/raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_DIR, "data/processed")
REPORTS_DIR = os.path.join(PROJECT_DIR, "reports")
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "model_final")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, REPORTS_DIR, CHECKPOINTS_DIR, FINAL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Arborescence créée sous:", PROJECT_DIR)


## 2. Dépôt du corpus

Dépose ton fichier `corpus.csv` (colonnes `french_text`, `fulfulde_text`) dans :
`Mon Drive/fr-fulfulde-mt/data/raw/corpus.csv`

Tu peux le faire via l'interface Drive avant de lancer ce notebook, ou décommenter la cellule
suivante pour l'uploader directement depuis ton ordinateur.


In [ ]:
# Option : upload direct depuis ton ordinateur (décommente si tu n'as pas déjà déposé le fichier)
# from google.colab import files
# uploaded = files.upload()
# import shutil
# for fname in uploaded.keys():
#     shutil.move(fname, os.path.join(DATA_RAW_DIR, "corpus.csv"))

RAW_CSV_PATH = os.path.join(DATA_RAW_DIR, "corpus.csv")
assert os.path.exists(RAW_CSV_PATH), f"Fichier introuvable: {RAW_CSV_PATH} -- dépose-le d'abord."
print("Corpus trouvé:", RAW_CSV_PATH)


## 3. Validation des données (non bloquante)

On vérifie l'encodage, les doublons, les lignes vides, les ratios de longueur suspects et les
caractères inattendus. **Rien n'est supprimé automatiquement** sauf les lignes 100% vides
(inutilisables des deux côtés). Le rapport complet est sauvegardé sur Drive pour que tu
l'examines à ton rythme — l'entraînement n'attend pas une correction manuelle pour démarrer.


In [ ]:
!pip install -q pandas

import sys
sys.path.insert(0, "/content")

# On récupère le script de validation depuis le dépôt cloné (voir cellule suivante)
# ou on le redéfinit inline si le dépôt n'est pas cloné dans cette session.


In [ ]:
# Clone du dépôt GitHub (remplace par l'URL de ton dépôt une fois créé)
# Si tu préfères travailler sans clone, les scripts sont aussi recopiés inline plus bas.
REPO_URL = "https://github.com/<ton-user>/fr-fulfulde-mt.git"  # <-- à adapter

if not os.path.exists("/content/fr-fulfulde-mt"):
    !git clone {REPO_URL} /content/fr-fulfulde-mt 2>/dev/null || echo "Clone échoué ou dépôt non encore public -- on utilise les scripts inline."

SCRIPTS_DIR = "/content/fr-fulfulde-mt/scripts" if os.path.exists("/content/fr-fulfulde-mt/scripts") else None
print("Scripts dir:", SCRIPTS_DIR)


In [ ]:
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

CLEAN_CSV_PATH = os.path.join(DATA_PROCESSED_DIR, "corpus_clean.csv")
REPORT_PATH = os.path.join(REPORTS_DIR, "validation_report.txt")

if SCRIPTS_DIR:
    validate_mod = load_module("validate_data", os.path.join(SCRIPTS_DIR, "validate_data.py"))
    df = validate_mod.load_csv(__import__("pathlib").Path(RAW_CSV_PATH))
    report = validate_mod.run_validation(df)
    validate_mod.write_report(report, __import__("pathlib").Path(REPORT_PATH))
    df_clean = df.drop(index=report["rows_to_drop"]).reset_index(drop=True)
    df_clean.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")
    print(f"{len(df_clean)} lignes conservées sur {len(df)}.")
else:
    raise RuntimeError("Clone le dépôt d'abord (cellule précédente), ou copie scripts/validate_data.py manuellement sur Drive.")

print()
print(open(REPORT_PATH, encoding='utf-8').read()[:3000])


## 4. Split train / val / test (90 / 5 / 5, seed fixe)

In [ ]:
import pandas as pd

split_mod = load_module("split_data", os.path.join(SCRIPTS_DIR, "split_data.py"))

df = pd.read_csv(CLEAN_CSV_PATH, encoding="utf-8")
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

n = len(df)
n_val = max(1, int(n * 0.05))
n_test = max(1, int(n * 0.05))
n_train = n - n_val - n_test

train_df = df.iloc[:n_train]
val_df = df.iloc[n_train:n_train + n_val]
test_df = df.iloc[n_train + n_val:]

train_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 5. Installation des dépendances ML


In [ ]:
!pip install -q transformers==4.* accelerate sentencepiece sacrebleu evaluate datasets

import torch
print("GPU disponible:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU seulement")


## 6. Chargement du modèle de base : NLLB-200-distilled-600M

On choisit le checkpoint **600M distillé** plutôt qu'un modèle plus gros : avec seulement
~4680 paires de phrases, un modèle plus grand (1.3B+) risquerait de sur-apprendre / d'oublier
ses capacités multilingues de base sans gain garanti. Le 600M offre le meilleur compromis
capacité / risque de surapprentissage pour ce volume de données, et tourne confortablement sur
un GPU T4 gratuit.

**Codes de langue utilisés :**
- Français : `fra_Latn`
- Fulfulde : `fuv_Latn` (seul code fulfulde existant dans NLLB-200 — couvre nativement le
  Nigerian Fulfulde, la variété la plus proche disponible du fulfulde Adamawa Cameroun).
  Le fine-tuning va spécialiser ce code vers ton dialecte à partir de tes données.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "fra_Latn"
TGT_LANG = "fuv_Latn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Modèle chargé sur", device)


## 7. Dataset bidirectionnel (FR→FF et FF→FR)

Chaque paire du corpus génère **deux exemples d'entraînement** (une direction et l'autre),
ce qui double le signal d'entraînement effectif (~9360 exemples) tout en partageant les mêmes
poids de modèle. C'est la stratégie recommandée pour ce volume de données limité.


In [ ]:
from torch.utils.data import Dataset

MAX_LENGTH = 128

class BidirectionalTranslationDataset(Dataset):
    def __init__(self, df, tokenizer, src_lang, tgt_lang, max_length=MAX_LENGTH):
        self.examples = []
        for _, row in df.iterrows():
            fr = str(row["french_text"]).strip()
            ff = str(row["fulfulde_text"]).strip()
            if not fr or not ff:
                continue
            # Direction 1: FR -> FF
            self.examples.append((fr, ff, src_lang, tgt_lang))
            # Direction 2: FF -> FR
            self.examples.append((ff, fr, tgt_lang, src_lang))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        src_text, tgt_text, src_lang, tgt_lang = self.examples[idx]
        self.tokenizer.src_lang = src_lang
        model_inputs = self.tokenizer(
            src_text, max_length=self.max_length, truncation=True
        )
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                tgt_text, max_length=self.max_length, truncation=True
            )
        model_inputs["labels"] = labels["input_ids"]
        # On force le token de langue cible pour forced_bos_token_id lors de la génération
        model_inputs["forced_bos_token_id"] = self.tokenizer.convert_tokens_to_ids(tgt_lang)
        return model_inputs


train_dataset = BidirectionalTranslationDataset(train_df, tokenizer, SRC_LANG, TGT_LANG)
val_dataset = BidirectionalTranslationDataset(val_df, tokenizer, SRC_LANG, TGT_LANG)
test_dataset = BidirectionalTranslationDataset(test_df, tokenizer, SRC_LANG, TGT_LANG)

print(f"Train: {len(train_dataset)} exemples (bidirectionnel)")
print(f"Val: {len(val_dataset)} exemples")
print(f"Test: {len(test_dataset)} exemples")


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)


## 8. Configuration de l'entraînement avec checkpointing Drive + reprise automatique

Les checkpoints sont écrits directement sur Drive (`CHECKPOINTS_DIR`). Si Colab se déconnecte
ou si tu relances ce notebook plus tard, la cellule ci-dessous **détecte automatiquement** le
dernier checkpoint et reprend l'entraînement à partir de là, sans rien perdre.


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.trainer_utils import get_last_checkpoint

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,          # garde les 3 derniers checkpoints (évite de saturer le Drive)
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   # batch effectif = 16
    num_train_epochs=15,             # petit corpus -> plus d'epochs, surveillées via eval_loss
    warmup_ratio=0.05,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
)

# Détection automatique d'un checkpoint existant sur Drive pour reprendre l'entraînement
last_checkpoint = None
if os.path.isdir(CHECKPOINTS_DIR) and os.listdir(CHECKPOINTS_DIR):
    last_checkpoint = get_last_checkpoint(CHECKPOINTS_DIR)
    if last_checkpoint:
        print(f"✅ Checkpoint trouvé, reprise depuis : {last_checkpoint}")
    else:
        print("Dossier de checkpoints présent mais vide/incomplet -- entraînement depuis le début.")
else:
    print("Aucun checkpoint trouvé -- entraînement depuis le début.")


## 9. Métriques d'évaluation (BLEU + chrF++)

chrF++ est généralement plus fiable que BLEU pour les langues morphologiquement riches /
peu dotées comme le fulfulde, car il travaille au niveau caractère et tolère mieux les petites
variations orthographiques. On rapporte les deux.


In [ ]:
import evaluate
import numpy as np

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    bleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels_bleu)
    chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels_bleu)

    return {
        "bleu": bleu_result["score"],
        "chrf++": chrf_result["score"],
    }


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Reprise automatique si un checkpoint existe, sinon entraînement depuis zéro
trainer.train(resume_from_checkpoint=last_checkpoint)


## 10. Évaluation finale sur le jeu de test (jamais vu pendant l'entraînement)


In [ ]:
test_results = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print(test_results)


## 11. Sauvegarde du modèle final sur Drive


In [ ]:
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Modèle final sauvegardé dans : {FINAL_MODEL_DIR}")

# On sauvegarde aussi les métriques de test pour le README / rapport
import json
with open(os.path.join(REPORTS_DIR, "test_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)


## 12. Démo d'inférence

Teste ici quelques phrases. **Fais valider ces traductions par un locuteur natif Adamawa**
avant de tirer des conclusions sur la qualité réelle.


In [ ]:
def translate(text, src_lang, tgt_lang, max_length=128):
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=max_length,
        num_beams=5,
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]


# Exemples -- remplace par tes propres phrases de test
examples_fr = [
    "Au commencement, Dieu créa les cieux et la terre.",
    "Aimez-vous les uns les autres.",
]

for ex in examples_fr:
    translation = translate(ex, SRC_LANG, TGT_LANG)
    print(f"FR : {ex}")
    print(f"FF : {translation}")
    print()


## Notes finales

- **Limite de dialecte** : `fuv_Latn` = Nigerian Fulfulde dans NLLB-200, pas un code Adamawa
  Cameroun dédié. La validation humaine par un locuteur natif Adamawa (toi, idéalement, ou
  quelqu'un d'autre de la zone) reste l'étape la plus importante avant de communiquer des
  résultats.
- **Limite de domaine** : corpus 100% biblique → bon sur ce registre, à valider séparément
  pour tout usage hors de ce domaine (conversation courante, administratif, etc.).
- **Prochaine étape suggérée** : une fois la faisabilité validée, élargir le corpus avec des
  données hors domaine biblique (actualités, conversation, etc.) pour viser un modèle plus
  général, comme discuté.
